# BDCC Midterm Review Notebook
### Frequent Itemset Mining · Neighborhood CF · Latent Factor CF

This notebook mirrors the three exercise notebooks, because the hands-on part of the midterm (40%) is based on them.

**How each topic works here**

| Icon | What you do |
| --- | --- |
| ✍️ **By hand** | Work it out on paper first, type your numbers into the dict, run the cell. It tells you ✅ or ❌. |
| 🔎 **Walkthrough** | Run it. It prints every step, the same way you'd write it by hand. |
| 💻 **Drill** | Write the function yourself (exam style). Replace `raise NotImplementedError`. |
| ✅ **Test** | Run it right after your drill. Same kind of `assert`s as the exercises. |
| 🔑 **Solution** | Click to expand. Try first! |

**Where each exercise problem is practiced**

| Exercise notebook | Problem | Drill here |
| --- | --- | --- |
| Frequent Itemset Mining | P2 `to_sparse` · P3 `brute` · P5 `apriori` · P6 `apriori2` · P7 `fpgrowth` · P8 `assoc` · P10 data prep | Part A |
| Neighborhood-based CF | P1 `raw_cos` · P2 `user_complete` · P3 `recommend` · P4 `item_complete` · P5 `dcg` · P6 `ndcg` | Part B |
| Latent factor CF | `als` · P1 `recommend_als` · P2 the $v_{ij}$ formula · P3 `cd` · P4 `recommend_cd` · Surprise | Part C |

Part D reruns the **original exercise tests on the Jester data** (only works on the course JupyterHub). Part E is a one-page exam-day cheat sheet.

**Run the setup cell first.** Keep your lines under 79 characters (PEP 8), since the graders flagged that before.

## Setup (run first)

In [ ]:
import itertools
import math
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
from scipy.linalg import lstsq
from scipy.sparse import csr_matrix
from numpy.testing import (assert_equal, assert_almost_equal,
                           assert_array_equal, assert_array_almost_equal)

pd.set_option("display.precision", 3)


def check(mine, truth):
    """Compare your paper answers with the computed truth."""
    for key, expected in truth.items():
        got = mine.get(key)
        if got is None:
            print(f"…  {key}: not answered yet")
            continue
        if isinstance(expected, (list, tuple)):
            good = list(got) == list(expected)
        else:
            tol = max(0.011, 0.01 * abs(float(expected)))
            good = abs(float(got) - float(expected)) <= tol
        if good:
            print(f"✅ {key} = {got}")
        else:
            if isinstance(expected, float):
                expected = round(expected, 3)
            print(f"❌ {key}: you wrote {got}, expected {expected}")


class H:
    """Show-your-work printers. You don't need to edit these."""

    # ---------- Frequent itemsets ----------
    @staticmethod
    def sup(db, items):
        items = set(items)
        return sum(1 for t in db if items <= set(t))

    @staticmethod
    def frequent(db, minsup):
        trans = [set(t) for t in db]
        items = sorted(set().union(*trans))
        out = {}
        for k in range(1, len(items) + 1):
            for c in itertools.combinations(items, k):
                s = sum(1 for t in trans if t.issuperset(c))
                if s >= minsup:
                    out[c] = s
        return out

    @staticmethod
    def rule(db, a, b):
        n = len(db)
        conf = H.sup(db, set(a) | set(b)) / H.sup(db, a)
        return conf, conf / (H.sup(db, b) / n)

    @staticmethod
    def trace_apriori(db, minsup):
        trans = [set(t) for t in db]
        level = [(i,) for i in sorted(set().union(*trans))]
        k = 1
        while level:
            print(f"--- Level {k}: count {len(level)} candidates ---")
            freq = []
            for c in level:
                s = sum(1 for t in trans if t.issuperset(c))
                ok = s >= minsup
                print(f"  {'+'.join(c):28s} sup={s}  "
                      f"{'keep' if ok else 'drop'}")
                if ok:
                    freq.append(c)
            fset, nxt = set(freq), []
            for a, b in itertools.combinations(sorted(freq), 2):
                if a[:-1] != b[:-1]:
                    continue
                cand = a + (b[-1],)
                bad = [s for s in itertools.combinations(cand, k)
                       if s not in fset]
                if bad:
                    print(f"  (skip {'+'.join(cand)}: part "
                          f"{'+'.join(bad[0])} already failed)")
                else:
                    nxt.append(cand)
            level, k = nxt, k + 1
        print("No candidates left → done.")

    @staticmethod
    def trace_projection(db, minsup):
        found = {}

        def grow(prefix, proj, depth):
            counts = Counter(i for t in proj for i in t)
            for item in sorted(counts):
                pad, new = "    " * depth, prefix + (item,)
                if counts[item] < minsup:
                    print(f"{pad}{'+'.join(new)}: sup={counts[item]}"
                          " (too rare, stop)")
                    continue
                sub = [sorted(i for i in t if i > item)
                       for t in proj if item in t]
                print(f"{pad}{'+'.join(new)}: sup={counts[item]} ✔  "
                      f"projected db = {sub}")
                found[new] = counts[item]
                grow(new, [set(s) for s in sub], depth + 1)

        grow((), [set(t) for t in db], 0)
        print(f"\n{len(found)} frequent itemsets found")

    # ---------- Neighborhood CF ----------
    @staticmethod
    def centered(df):
        return df.sub(df.mean(axis=1), axis=0)

    @staticmethod
    def _cos(a, b):
        m = a.notna() & b.notna()
        if not m.any():
            return 0.0
        a, b = a[m].astype(float), b[m].astype(float)
        d = np.linalg.norm(a) * np.linalg.norm(b)
        return 0.0 if d == 0 else float(a @ b / d)

    @staticmethod
    def adj_cos(df, u, v):
        c = H.centered(df)
        return H._cos(c.loc[u], c.loc[v])

    @staticmethod
    def _explain(mu_u, sims, ratings, k, positive_only, what):
        pool = sims[sims > 0] if positive_only else sims
        top = (pool.sort_index()
               .sort_values(ascending=False, kind="mergesort").iloc[:k])
        rule = ("top-k among POSITIVE similarities (slides)"
                if positive_only else "top-k overall (exercise code)")
        print(f"Similarities: {sims.round(3).to_dict()}")
        print(f"Neighbors, {rule}: {top.round(3).to_dict()}")
        used = top[ratings[top.index].notna()]
        if used.empty or used.sum() == 0:
            print(f"None of them {what} → prediction is NaN")
            return np.nan
        parts = " + ".join(f"({s:.3f})({ratings[j]:+.3f})"
                           for j, s in used.items())
        num = float((used * ratings[used.index]).sum())
        den = float(used.sum())
        print(f"Used ({what}): {used.round(3).to_dict()}")
        print(f"centered guess = [{parts}] / {den:.3f} = {num / den:+.3f}")
        print(f"final guess    = {mu_u:.3f} + ({num / den:+.3f}) "
              f"= {mu_u + num / den:.3f}")
        return mu_u + num / den

    @staticmethod
    def explain_user(df, user, item, k, positive_only=False):
        mu, c = df.mean(axis=1), H.centered(df)
        print(f"Average of {user} = {mu[user]:.3f}")
        sims = pd.Series({v: H._cos(c.loc[user], c.loc[v])
                          for v in df.index if v != user})
        return H._explain(mu[user], sims, c[item], k, positive_only,
                          f"rated {item}")

    @staticmethod
    def explain_item(df, user, item, k, positive_only=False):
        mu, c = df.mean(axis=1), H.centered(df)
        print(f"Average of {user} = {mu[user]:.3f}")
        sims = pd.Series({j: H._cos(c[item], c[j])
                          for j in df.columns if j != item})
        return H._explain(mu[user], sims, c.loc[user], k, positive_only,
                          f"rated by {user}")

    @staticmethod
    def explain_dcg(ratings, recos):
        r = pd.Series(ratings, dtype=float)

        def table(order, title):
            rows, total = [], 0.0
            for pos, it in enumerate(order, start=1):
                rel = r.get(it, np.nan)
                rel = 0.0 if pd.isna(rel) else rel
                gain, disc = 2 ** rel - 1, math.log2(pos + 1)
                total += gain / disc
                rows.append((pos, it, rel, gain, round(disc, 3),
                             round(gain / disc, 2)))
            print(title)
            cols = ["pos", "item", "rating", "2^r-1", "log2(pos+1)",
                    "gain/disc"]
            print(pd.DataFrame(rows, columns=cols).to_string(index=False))
            print(f"  total = {total:.2f}\n")
            return total

        dcg = table(recos, "Your list")
        best = (r.dropna().sort_index()
                .sort_values(ascending=False, kind="mergesort"))
        idcg = table(list(best.index[:len(recos)]), "Ideal list")
        print(f"NDCG = {dcg:.2f} / {idcg:.2f} = {dcg / idcg:.3f}")
        return dcg / idcg

    # ---------- Latent factors ----------
    @staticmethod
    def trace_als(M, d, rounds=1):
        M = np.asarray(M, dtype=float)
        U, V = np.ones((M.shape[0], d)), np.ones((M.shape[1], d))
        for r in range(rounds):
            for j in range(V.shape[0]):
                has = np.isfinite(M[:, j])
                V[j] = lstsq(U[has], M[has, j])[0]
            print(f"Round {r + 1}: V transposed (U held fixed)")
            print(np.round(V.T, 4))
            for i in range(U.shape[0]):
                has = np.isfinite(M[i, :])
                U[i] = lstsq(V[has], M[i, has])[0]
            print(f"Round {r + 1}: U (V held fixed)")
            print(np.round(U, 4))
            rmse = np.sqrt(np.nanmean((M - U @ V.T) ** 2))
            print(f"RMSE on known ratings = {rmse:.4f}\n")
        return U, V

    @staticmethod
    def best_u(M, U, V, i, j):
        has = np.isfinite(M[i])
        rest = U[i] @ V[has].T - U[i, j] * V[has, j]
        return V[has, j] @ (M[i, has] - rest) / (V[has, j] @ V[has, j])

    @staticmethod
    def best_v(M, U, V, i, j):
        has = np.isfinite(M[:, i])
        rest = U[has] @ V[i] - U[has, j] * V[i, j]
        return U[has, j] @ (M[has, i] - rest) / (U[has, j] @ U[has, j])

    @staticmethod
    def trace_cd(M, d, steps=4):
        M = np.asarray(M, dtype=float)
        U, V = np.ones((M.shape[0], d)), np.ones((M.shape[1], d))
        order = ([("U", i, j) for i in range(U.shape[0])
                  for j in range(d)]
                 + [("V", i, j) for i in range(V.shape[0])
                    for j in range(d)])
        print(f"start SSE = {np.nansum((M - U @ V.T) ** 2):.3f}")
        for name, i, j in order[:steps]:
            if name == "U":
                U[i, j] = new = H.best_u(M, U, V, i, j)
            else:
                V[i, j] = new = H.best_v(M, U, V, i, j)
            sse = np.nansum((M - U @ V.T) ** 2)
            print(f"update {name}[{i + 1},{j + 1}] = {new:.4f}   "
                  f"SSE = {sse:.3f}")
        return U, V


print("Setup done ✔")

---
# Part A — Frequent Itemset Mining

The 5-receipt database from the slides and notebook:

In [ ]:
db = [{"bread", "butter", "milk"},
      {"eggs", "milk", "yogurt"},
      {"bread", "cheese", "eggs", "milk"},
      {"eggs", "milk", "yogurt"},
      {"cheese", "milk", "yogurt"}]
N = len(db)

### ✍️ A1. Support by hand

$$\text{sup}(X) = \#\{\text{receipts containing all of } X\} \qquad \text{relSup}(X) = \frac{\text{sup}(X)}{N}$$

Count on paper, then fill in the numbers.

In [ ]:
my = {
    "sup(bread, milk)": None,
    "sup(eggs, yogurt)": None,
    "sup(cheese, milk, yogurt)": None,
    "relSup(milk)": None,
    "relSup(eggs, milk)": None,
    "# of frequent itemsets at minsup = 2": None,
}
check(my, {
    "sup(bread, milk)": H.sup(db, {"bread", "milk"}),
    "sup(eggs, yogurt)": H.sup(db, {"eggs", "yogurt"}),
    "sup(cheese, milk, yogurt)": H.sup(db, {"cheese", "milk", "yogurt"}),
    "relSup(milk)": H.sup(db, {"milk"}) / N,
    "relSup(eggs, milk)": H.sup(db, {"eggs", "milk"}) / N,
    "# of frequent itemsets at minsup = 2": len(H.frequent(db, 2)),
})

### 💻 A2. Warm-up: `support`

In [ ]:
def support(db, itemset):
    """Number of transactions that contain every item in itemset."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert_equal(support(db, {"bread", "milk"}), 2)
assert_equal(support(db, ["milk"]), 5)
assert_equal(support(db, ("eggs", "yogurt")), 2)
assert_equal(support(db, {"butter", "eggs"}), 0)
print("✅ support passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def support(db, itemset):
    """Number of transactions that contain every item in itemset."""
    need = set(itemset)
    return sum(1 for t in db if need <= set(t))
```

</details>

### 💻 A3. `to_sparse` (exercise Problem 2)
Rows = transactions, columns = items **sorted alphabetically**, 1 = present.

In [ ]:
def to_sparse(db):
    """Return a CSR matrix (transactions x alphabetically sorted items)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
db_sparse = to_sparse(db)
assert isinstance(db_sparse, csr_matrix)
assert_equal(db_sparse.shape, (5, 6))
assert_array_equal(db_sparse[0, :].toarray().squeeze(), [1, 1, 0, 0, 1, 0])
assert_array_equal(db_sparse[-1, :].toarray().squeeze(), [0, 0, 1, 0, 1, 1])
print("✅ to_sparse passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def to_sparse(db):
    """Return a CSR matrix (transactions x alphabetically sorted items)."""
    trans = [set(t) for t in db]
    items = sorted(set().union(*trans))
    col = {item: j for j, item in enumerate(items)}
    rows, cols = [], []
    for r, t in enumerate(trans):
        for item in t:
            rows.append(r)
            cols.append(col[item])
    data = np.ones(len(rows), dtype=np.int8)
    return csr_matrix((data, (rows, cols)),
                      shape=(len(trans), len(items)))
```

</details>

### 💻 A4. `brute` (exercise Problem 3)

Check **every** non-empty group of items. With $d$ items that's $2^d - 1$ groups, so the work is about $N \times 2^d$.

**Output format used by every FIM function in the exercises:** a list of `(tuple_of_items, support)`, items sorted alphabetically inside each tuple, and the list sorted by **decreasing support → decreasing size → alphabetical**:

```python
sorted(counts.items(), key=lambda kv: (-kv[1], -len(kv[0]), kv[0]))
```
Use only the standard library.

In [ ]:
def brute(db, minsup):
    """All frequent itemsets by checking every possible itemset."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
fi_brute = brute(db, 2)
assert_equal(len(fi_brute), 11)
assert_equal(fi_brute[0], (("milk",), 5))
assert_equal(fi_brute[-1], (("cheese",), 2))
print("✅ brute passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def brute(db, minsup):
    """All frequent itemsets by checking every possible itemset."""
    trans = [set(t) for t in db]
    items = sorted(set().union(*trans))
    counts = {}
    for k in range(1, len(items) + 1):
        for combo in itertools.combinations(items, k):
            sup = sum(1 for t in trans if t.issuperset(combo))
            if sup >= minsup:
                counts[combo] = sup
    return sorted(counts.items(),
                  key=lambda kv: (-kv[1], -len(kv[0]), kv[0]))
```

</details>

### 🔎 A5. Apriori, level by level

Rule behind Apriori: if a group fails, **every bigger group containing it fails too** (downward closure). So we only grow winners:
1. count the singles, keep the frequent ones;
2. join two winners of size $k-1$ that share their first $k-2$ items;
3. skip a candidate if any of its $(k-1)$-parts failed;
4. count the rest, repeat.

In [ ]:
H.trace_apriori(db, 2)

### 💻 A6. `apriori` (exercise Problem 5)

In [ ]:
def apriori(db, minsup):
    """All frequent itemsets using the Apriori algorithm."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
fi_apriori = apriori(db, 2)
assert_equal(len(fi_apriori), 11)
assert_equal(fi_apriori[0], (("milk",), 5))
assert_equal(fi_apriori[-1], (("cheese",), 2))
assert_equal(fi_apriori, brute(db, 2))
print("✅ apriori passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def apriori(db, minsup):
    """All frequent itemsets using the Apriori algorithm."""
    trans = [set(t) for t in db]
    items = sorted(set().union(*trans))

    def count(candidates):
        out = {}
        for c in candidates:
            sup = sum(1 for t in trans if t.issuperset(c))
            if sup >= minsup:
                out[c] = sup
        return out

    freq = count((i,) for i in items)
    result = dict(freq)
    k = 2
    while freq:
        prev = sorted(freq)
        prev_set = set(prev)
        candidates = []
        for a, b in itertools.combinations(prev, 2):
            if a[:k - 2] != b[:k - 2]:        # join step
                continue
            cand = a + (b[-1],)
            subsets = itertools.combinations(cand, k - 1)
            if all(s in prev_set for s in subsets):   # prune step
                candidates.append(cand)
        freq = count(candidates)
        result.update(freq)
        k += 1
    return sorted(result.items(),
                  key=lambda kv: (-kv[1], -len(kv[0]), kv[0]))
```

</details>

### 💻 A7. `apriori2`: maximal itemsets only (exercise Problem 6)
**Maximal** = frequent, and no frequent superset exists.

In [ ]:
def apriori2(db, minsup):
    """Maximal frequent itemsets (frequent, with no frequent superset)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
maxfi = apriori2(db, 2)
assert_equal(len(maxfi), 3)
assert_equal(maxfi, [(("eggs", "milk", "yogurt"), 2),
                     (("bread", "milk"), 2),
                     (("cheese", "milk"), 2)])
print("✅ apriori2 passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def apriori2(db, minsup):
    """Maximal frequent itemsets (frequent, with no frequent superset)."""
    fi = dict(apriori(db, minsup))
    maximal = {c: s for c, s in fi.items()
               if not any(set(c) < set(other) for other in fi)}
    return sorted(maximal.items(),
                  key=lambda kv: (-kv[1], -len(kv[0]), kv[0]))
```

</details>

### 🔎 A8. FP-Growth as the slides do it: projected databases

Pick an item, keep only the receipts containing it, and keep only the items that come **after** it (alphabetically). That smaller pile is the **projected database**. Recurse on it.

In [ ]:
H.trace_projection(db, 2)

### 💻 A9. `fpgrowth` (exercise Problem 7)

The recursive projected-database version below is short enough to write under exam time and gives the same answers as the FP-tree version. Standard library only.

In [ ]:
def fpgrowth(db, minsup):
    """All frequent itemsets by pattern growth on projected databases."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
fi_fp = fpgrowth(db, 2)
assert_equal(len(fi_fp), 11)
assert_equal(fi_fp[0], (("milk",), 5))
assert_equal(fi_fp[-1], (("cheese",), 2))

random.seed(1)                      # bigger random check vs brute force
letters = list("abcdefgh")
big = [set(random.sample(letters, random.randint(1, 5)))
       for _ in range(40)]
assert_equal(fpgrowth(big, 5), brute(big, 5))
print("✅ fpgrowth passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def fpgrowth(db, minsup):
    """All frequent itemsets by pattern growth on projected databases."""
    out = {}

    def grow(prefix, projected):
        counts = Counter(i for t in projected for i in t)
        for item in sorted(counts):
            if counts[item] < minsup:
                continue
            new = prefix + (item,)
            out[tuple(sorted(new))] = counts[item]
            # keep receipts with `item`, and only the items after it
            sub = [{i for i in t if i > item}
                   for t in projected if item in t]
            grow(new, sub)

    grow((), [set(t) for t in db])
    return sorted(out.items(),
                  key=lambda kv: (-kv[1], -len(kv[0]), kv[0]))
```

</details>

### ✍️ A10. Confidence and lift by hand

$$\text{conf}(A \rightarrow B) = \frac{\text{sup}(A \cup B)}{\text{sup}(A)} \qquad \text{lift}(A \rightarrow B) = \frac{\text{conf}(A \rightarrow B)}{\text{relSup}(B)}$$

Lift > 1: go together · lift = 1: no link · lift < 1: avoid each other.

In [ ]:
my = {
    "conf(yogurt -> milk)": None,
    "conf(milk -> yogurt)": None,
    "lift(eggs, milk -> yogurt)": None,
    "lift(bread -> butter)": None,
    "lift(bread -> eggs)": None,
}
check(my, {
    "conf(yogurt -> milk)": H.rule(db, {"yogurt"}, {"milk"})[0],
    "conf(milk -> yogurt)": H.rule(db, {"milk"}, {"yogurt"})[0],
    "lift(eggs, milk -> yogurt)":
        H.rule(db, {"eggs", "milk"}, {"yogurt"})[1],
    "lift(bread -> butter)": H.rule(db, {"bread"}, {"butter"})[1],
    "lift(bread -> eggs)": H.rule(db, {"bread"}, {"eggs"})[1],
})

### 💻 A11. `assoc` (exercise Problem 8)

Input: the frequent-itemset list from your functions, `minconf`, and the number of transactions. For each itemset of size ≥ 2, make **each single item** the consequent and the rest the antecedent. Return dicts with keys `antecedent` (tuple), `consequent` (item), `support`, `confidence`, `lift`. Sort by decreasing lift, confidence, support, then consequent, then antecedent.

In [ ]:
def assoc(fi, minconf, db_size):
    """Association rules with a single-item consequent."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
rules = assoc(
    [(("a",), 5), (("a", "b"), 3), (("a", "c"), 3), (("b",), 3),
     (("c",), 3), (("a", "b", "c"), 2), (("a", "d"), 2), (("a", "e"), 2),
     (("b", "c"), 2), (("d",), 2), (("e",), 2)],
    0.2, 5)
assert_equal(len(rules), 13)
assert_equal(rules[0], {"antecedent": ("a", "c"), "consequent": "b",
                        "support": 2, "confidence": 0.6666666666666666,
                        "lift": 1.1111111111111112})
assert_equal(rules[-1], {"antecedent": ("a",), "consequent": "e",
                         "support": 2, "confidence": 0.4, "lift": 1.0})
print("✅ assoc passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def assoc(fi, minconf, db_size):
    """Association rules with a single-item consequent."""
    sup = dict(fi)
    rules = []
    for itemset, s in fi:
        if len(itemset) < 2:
            continue
        for cons in itemset:
            ante = tuple(i for i in itemset if i != cons)
            conf = s / sup[ante]
            if conf < minconf:
                continue
            lift = conf / (sup[(cons,)] / db_size)
            rules.append({"antecedent": ante, "consequent": cons,
                          "support": s, "confidence": conf,
                          "lift": lift})
    return sorted(rules, key=lambda r: (-r["lift"], -r["confidence"],
                                        -r["support"], r["consequent"],
                                        r["antecedent"]))
```

</details>

### 💻 A12. Variant: the slides' version with multi-item consequents

The slides split each itemset **every** way, e.g. `eggs → {yogurt, milk}`. Return tuples `(antecedent, consequent, support, confidence, lift)`, both sides as sorted tuples. At minsup 2 and minconf 0.6 the slides get **13 rules**.

In [ ]:
def assoc_all(fi, minconf, db_size):
    """Rules from every split of each itemset (multi-item consequents)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
slide_rules = assoc_all(apriori(db, 2), 0.6, 5)
pairs = {(a, c) for a, c, *_ in slide_rules}
assert_equal(len(slide_rules), 13)
assert (("eggs",), ("milk", "yogurt")) in pairs
assert (("milk",), ("eggs", "yogurt")) not in pairs      # conf 0.4
assert (("milk",), ("bread",)) not in pairs              # conf 0.4
print("✅ assoc_all passed")
for a, c, s, conf, lift in slide_rules:
    print(f"{str(a):22s} -> {str(c):20s} conf={conf:.2f} lift={lift:.2f}")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def assoc_all(fi, minconf, db_size):
    """Rules from every split of each itemset (multi-item consequents)."""
    sup = dict(fi)
    rules = []
    for itemset, s in fi:
        for r in range(1, len(itemset)):
            for ante in itertools.combinations(itemset, r):
                cons = tuple(i for i in itemset if i not in ante)
                conf = s / sup[ante]
                if conf >= minconf:
                    lift = conf / (sup[cons] / db_size)
                    rules.append((ante, cons, s, conf, lift))
    return sorted(rules, key=lambda r: (-r[3], -r[2], r[0], r[1]))
```

</details>

### ✍️ A13. Fresh by-hand drill (new numbers)

6 café receipts, **minsup = 3**, **minconf = 0.7**. On paper: run Apriori, list the maximal itemsets, then find every single-consequent rule with confidence ≥ 0.7 and its lift.

In [ ]:
cafe = [{"coffee", "donut", "milk"},
        {"coffee", "donut"},
        {"coffee", "milk", "sugar"},
        {"donut", "milk"},
        {"coffee", "donut", "milk", "sugar"},
        {"coffee", "sugar"}]

my = {
    "sup(coffee, donut, milk)": None,
    "# frequent itemsets": None,
    "# maximal itemsets": None,
    "conf(donut -> coffee)": None,
    "lift(donut -> coffee)": None,
    "lift(sugar -> coffee)": None,
    "# rules kept": None,
}
fi_cafe = H.frequent(cafe, 3)
n_max = sum(1 for c in fi_cafe
            if not any(set(c) < set(o) for o in fi_cafe))
n_rules = sum(1 for c in fi_cafe if len(c) >= 2 for i in c
              if H.rule(cafe, set(c) - {i}, {i})[0] >= 0.7)
check(my, {
    "sup(coffee, donut, milk)": H.sup(cafe, {"coffee", "donut", "milk"}),
    "# frequent itemsets": len(fi_cafe),
    "# maximal itemsets": n_max,
    "conf(donut -> coffee)": H.rule(cafe, {"donut"}, {"coffee"})[0],
    "lift(donut -> coffee)": H.rule(cafe, {"donut"}, {"coffee"})[1],
    "lift(sugar -> coffee)": H.rule(cafe, {"sugar"}, {"coffee"})[1],
    "# rules kept": n_rules,
})

In [ ]:
# Then confirm with your own functions:
H.trace_apriori(cafe, 3)
print(apriori2(cafe, 3))
for r in assoc(apriori(cafe, 3), 0.7, len(cafe)):
    print(r)

### 💻 A14. Data prep like Problem 10 (Online Retail)

Turn a POS table into a list of transactions: drop cancelled invoices (`InvoiceNo` starting with `C`), drop quantities ≤ 0, drop missing descriptions, strip spaces, then group by invoice. Return each invoice's items **sorted and without duplicates**, invoices in order.

In [ ]:
def to_db(df):
    """POS DataFrame -> list of sorted item lists, one per invoice."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
toy = pd.DataFrame({
    "InvoiceNo": ["536365", "536365", "536366", "C536367", "536368",
                  "536368", "536369", "536366"],
    "Description": [" MUG ", "PLATE", "MUG", "MUG", "BOWL", None,
                    "PLATE", "MUG"],
    "Quantity": [2, 1, 3, 1, 4, 1, -2, 1],
})
assert_equal(to_db(toy), [["MUG", "PLATE"], ["MUG"], ["BOWL"]])
print("✅ to_db passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def to_db(df):
    """POS DataFrame -> list of sorted item lists, one per invoice."""
    df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
    df = df[df["Quantity"] > 0]
    df = df.dropna(subset=["Description"])
    df = df.assign(Description=df["Description"].str.strip())
    return (df.groupby("InvoiceNo")["Description"]
              .apply(lambda s: sorted(set(s))).tolist())
```

</details>

### A15. Optional: runtime check (Problems 4 and 9)

Brute force grows like $2^d$ in the number of items $d$ but only linearly in $N$. Run this after your three functions work.

In [ ]:
def make_db(n_trans, n_items, t_size=4, seed=0):
    rng = random.Random(seed)
    items = [f"i{j:02d}" for j in range(n_items)]
    return [set(rng.sample(items, min(t_size, n_items)))
            for _ in range(n_trans)]


def best_time(fn, *args):
    t = time.perf_counter()
    fn(*args)
    return time.perf_counter() - t


print("items  brute   apriori  fpgrowth")
for d in [6, 8, 10, 12]:
    small = make_db(200, d)
    print(f"{d:5d} {best_time(brute, small, 20):7.3f} "
          f"{best_time(apriori, small, 20):8.3f} "
          f"{best_time(fpgrowth, small, 20):9.3f}")

### A16. Optional: pyFIM

In `fim`, a **positive** `supp` is a **percentage**; a **negative** `supp` is an absolute count. `zmin` = minimum itemset size. `target="r"` returns rules; `report="l"` adds lift.

In [ ]:
try:
    import fim
    ours = sorted(((tuple(sorted(s)), c)
                   for s, c in fim.fpgrowth(db, supp=-2)),
                  key=lambda kv: (-kv[1], -len(kv[0]), kv[0]))
    assert_equal(ours, brute(db, 2))
    print("pyFIM agrees with brute ✔")
except ImportError:
    print("pyFIM not installed here: pip install pyfim")

---
# Part B — Neighborhood-based Collaborative Filtering

In [ ]:
nan = np.nan

# The exercise's HP/TW/SW table (the code puts D's two 3s under HP2, SW3)
df_utility = pd.DataFrame(
    {"HP1": [4, 5, nan, nan], "HP2": [nan, 5, nan, 3],
     "HP3": [nan, 4, nan, nan], "TW": [5, nan, 2, nan],
     "SW1": [1, nan, 4, nan], "SW2": [nan, nan, 5, nan],
     "SW3": [nan, nan, nan, 3]},
    index=list("ABCD"))

# The slides' 5 users x 6 items table
df_six = pd.DataFrame(
    {"a": [7, 6, nan, 1, 1], "b": [6, 7, 3, 2, nan],
     "c": [7, nan, 3, 2, 1], "d": [4, 4, 1, 3, 2],
     "e": [5, 3, 1, 3, 3], "f": [4, 4, nan, 4, 3]},
    index=list("ABCDE"))

# A fresh table for by-hand practice
df_four = pd.DataFrame(
    {"i1": [5, 4, 1, 5], "i2": [4, 5, 1, 3], "i3": [nan, 5, 2, 4],
     "i4": [1, 1, 5, 2], "i5": [2, nan, 4, 1]},
    index=list("PQRS"))
df_six

### ✍️ B1. Means, centering and similarity by hand

1. Average each user's **own** ratings: $\mu_u$.
2. Center: $r_{ui} - \mu_u$.
3. Compare two users **only on items both rated**:

$$\text{sim}(u,v) = \frac{\sum_i (r_{ui}-\mu_u)(r_{vi}-\mu_v)}{\sqrt{\sum_i (r_{ui}-\mu_u)^2}\,\sqrt{\sum_i (r_{vi}-\mu_v)^2}}$$

In [ ]:
my = {
    "mu_A (df_utility)": None,
    "sim(A, C) adjusted (df_utility)": None,
    "mu_P (df_four)": None,
    "sim(P, S) (df_four)": None,
    "sim(P, Q) (df_four)": None,
    "sim(P, R) (df_four)": None,
}
check(my, {
    "mu_A (df_utility)": df_utility.loc["A"].mean(),
    "sim(A, C) adjusted (df_utility)": H.adj_cos(df_utility, "A", "C"),
    "mu_P (df_four)": df_four.loc["P"].mean(),
    "sim(P, S) (df_four)": H.adj_cos(df_four, "P", "S"),
    "sim(P, Q) (df_four)": H.adj_cos(df_four, "P", "Q"),
    "sim(P, R) (df_four)": H.adj_cos(df_four, "P", "R"),
})

### 💻 B2. `raw_cos` (exercise Problem 1)

Cosine similarity of two Series using **only positions where both are non-null**. Return 0 if nothing is shared or a vector has length 0. Feed it **centered** rows and it becomes the adjusted cosine.

In [ ]:
def raw_cos(a, b):
    """Cosine similarity over the common non-null entries of a and b."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert_almost_equal(raw_cos(df_utility.loc["A"], df_utility.loc["B"]), 1)
assert_almost_equal(raw_cos(df_utility.loc["A"], df_utility.loc["C"]),
                    0.6139406135149205)
assert_almost_equal(raw_cos(df_utility.loc["A"], df_utility.loc["D"]), 0)
centered = df_utility.sub(df_utility.mean(axis=1), axis=0)
assert_almost_equal(raw_cos(centered.loc["A"], centered.loc["C"]),
                    -0.7295372041400852)
print("✅ raw_cos passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def raw_cos(a, b):
    """Cosine similarity over the common non-null entries of a and b."""
    if len(a) == 0 or len(b) == 0:
        return 0
    mask = a.notna() & b.notna()
    if not mask.any():
        return 0
    a, b = a[mask].astype(float), b[mask].astype(float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0
    return float(np.dot(a, b) / denom)
```

</details>

### 🔎 B3. One user-based prediction, step by step

$$\hat{r}_{ui} = \mu_u + \frac{\sum_{v} \text{sim}(u,v)\,(r_{vi}-\mu_v)}{\sum_{v} \text{sim}(u,v)}$$

⚠️ **Two neighbor rules.**
- **Exercise code:** take the top $k$ users by similarity (even negative ones), then keep those who rated the item.
- **Slides' hand examples:** only use neighbors with **positive** similarity.

On the slides' table they give different answers for B's rating of c. Compare:

In [ ]:
print("EXERCISE RULE")
H.explain_user(df_six, "B", "c", k=3)
print("\nSLIDES RULE")
H.explain_user(df_six, "B", "c", k=3, positive_only=True)

The negative neighbor D pushes the exercise-rule guess to 7.35, above the scale. With Jester and $k = 5$, the top neighbors are usually all positive, so both rules tend to agree there. **In the exam, follow the rule the question describes.**

Your by-hand table: both top-2 neighbors of P are positive, so the rules agree.

In [ ]:
H.explain_user(df_four, "P", "i3", k=2)

### 💻 B4. `user_complete` (exercise Problem 2)

Mean-center, compute user-user adjusted cosine, then fill every blank. Break similarity ties by user id (`sort_index()` first, then a **stable** `sort_values(kind="mergesort")`). If no chosen neighbor rated the item (weight sum 0), leave NaN.

Add the optional `positive_only` flag so you can reproduce the slides too.

In [ ]:
def user_complete(df_utility, k, positive_only=False):
    """Fill the utility matrix with user-based CF (adjusted cosine)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
four = user_complete(df_four, 2)
assert_almost_equal(four.loc["P", "i3"], 4.1215, decimal=3)
pd.testing.assert_frame_equal(four.where(df_four.notna()), df_four,
                              check_dtype=False)    # knowns unchanged

slides = user_complete(df_six, 3, positive_only=True)
assert_array_almost_equal(
    [slides.loc["B", "c"], slides.loc["C", "a"], slides.loc["C", "f"],
     slides.loc["E", "b"]],
    [6.0137, 3.3464, 0.8584, 1.5], decimal=3)

exercise = user_complete(df_six, 3)
assert_almost_equal(exercise.loc["E", "b"], 4.1733, decimal=3)
print("✅ user_complete passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def user_complete(df_utility, k, positive_only=False):
    """Fill the utility matrix with user-based CF (adjusted cosine)."""
    means = df_utility.mean(axis=1)
    centered = df_utility.sub(means, axis=0)
    users = df_utility.index
    sims = pd.DataFrame(0.0, index=users, columns=users)
    for i, u in enumerate(users):
        for v in users[i + 1:]:
            s = raw_cos(centered.loc[u], centered.loc[v])
            sims.loc[u, v] = sims.loc[v, u] = s

    done = df_utility.copy()
    for u in users:
        pool = sims.loc[u].drop(u)
        if positive_only:
            pool = pool[pool > 0]
        nbrs = (pool.sort_index()
                .sort_values(ascending=False, kind="mergesort")
                .iloc[:k])
        for item in df_utility.columns[df_utility.loc[u].isna()]:
            nb = nbrs[centered.loc[nbrs.index, item].notna()]
            if nb.sum() == 0:
                done.loc[u, item] = np.nan
            else:
                done.loc[u, item] = means[u] + (
                    (nb * centered.loc[nb.index, item]).sum() / nb.sum())
    return done
```

</details>

### 💻 B5. `recommend` (exercise Problem 3)
Top `n` **unrated** items for `user`, by predicted rating (high → low), ties by item id. Drop NaN predictions.

In [ ]:
def recommend(user, df_utility, df_completed, n):
    """Top-n unrated items for user, ties broken by item id."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert_equal(recommend("C", df_six, slides, 2), ["a", "f"])
orig = pd.DataFrame({"a": [1.0], "b": [nan], "c": [nan], "d": [nan],
                     "e": [nan]}, index=["X"])
done = pd.DataFrame({"a": [9.0], "b": [3.0], "c": [3.0], "d": [4.0],
                     "e": [nan]}, index=["X"])
assert_equal(recommend("X", orig, done, 3), ["d", "b", "c"])  # tie b, c
print("✅ recommend passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def recommend(user, df_utility, df_completed, n):
    """Top-n unrated items for user, ties broken by item id."""
    unrated = df_utility.columns[df_utility.loc[user].isna()]
    preds = df_completed.loc[user, unrated].dropna()
    ranked = (preds.sort_index()
              .sort_values(ascending=False, kind="mergesort"))
    return ranked.index[:n].tolist()
```

</details>

### 🔎 B6. One item-based prediction, step by step

$$\hat{r}_{ui} = \mu_u + \frac{\sum_{j} \text{sim}(i,j)\,(r_{uj}-\mu_u)}{\sum_{j} \text{sim}(i,j)}$$

Same idea on **columns**: find the $k$ items most like item $i$, then average **your own** centered ratings on the ones you rated. Item similarity is still computed on the matrix centered by **user** means.

In [ ]:
H.explain_item(df_six, "B", "c", k=2)

### 💻 B7. `item_complete` (exercise Problem 4)

In [ ]:
def item_complete(df_utility, k, positive_only=False):
    """Fill the utility matrix with item-based CF (adjusted cosine)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
six = item_complete(df_six, 2)
assert_array_almost_equal(
    [six.loc["C", "a"], six.loc["E", "b"], six.loc["B", "c"],
     six.loc["C", "f"]],
    [3.0, 1.0, 6.4891, 1.0], decimal=3)
hp = item_complete(df_utility, 2)
assert_array_almost_equal(
    [hp.loc["C", "HP1"], hp.loc["A", "HP2"], hp.loc["B", "TW"],
     hp.loc["A", "SW2"]],
    [2.0, 4.0, 5.0, 1.0], decimal=3)
print("✅ item_complete passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def item_complete(df_utility, k, positive_only=False):
    """Fill the utility matrix with item-based CF (adjusted cosine)."""
    means = df_utility.mean(axis=1)
    centered = df_utility.sub(means, axis=0)       # centered by USER mean
    items = df_utility.columns
    sims = pd.DataFrame(0.0, index=items, columns=items)
    for i, a in enumerate(items):
        for b in items[i + 1:]:
            s = raw_cos(centered[a], centered[b])
            sims.loc[a, b] = sims.loc[b, a] = s

    done = df_utility.copy()
    for item in items:
        pool = sims.loc[item].drop(item)
        if positive_only:
            pool = pool[pool > 0]
        nbrs = (pool.sort_index()
                .sort_values(ascending=False, kind="mergesort")
                .iloc[:k])
        for u in df_utility.index[df_utility[item].isna()]:
            nb = nbrs[centered.loc[u, nbrs.index].notna()]
            if nb.sum() == 0:
                done.loc[u, item] = np.nan
            else:
                done.loc[u, item] = means[u] + (
                    (nb * centered.loc[u, nb.index]).sum() / nb.sum())
    return done
```

</details>

### ✍️ B8. DCG and NDCG by hand

$$\text{DCG} = \frac{1}{m}\sum_{u=1}^{m} \sum_{j} \frac{2^{r_{uj}} - 1}{\log_2(v_j + 1)} \qquad \text{NDCG} = \frac{\text{DCG}}{\text{IDCG}}$$

$v_j$ = position in the list (starts at 1). A recommended item the user never rated counts as 0. IDCG uses the user's own top ratings in the ideal order.

Fresh drill: true ratings W = 4, X = 1, Y = 3, Z = 2; the list shown was **[X, W, Y]**.

In [ ]:
drill_ratings = {"W": 4, "X": 1, "Y": 3, "Z": 2}
my = {"DCG": None, "IDCG": None, "NDCG": None}
_g = lambda order: sum((2 ** drill_ratings[it] - 1) / math.log2(p + 2)
                       for p, it in enumerate(order))
check(my, {"DCG": _g(["X", "W", "Y"]), "IDCG": _g(["W", "Y", "Z"]),
           "NDCG": _g(["X", "W", "Y"]) / _g(["W", "Y", "Z"])})

In [ ]:
# 🔎 The slides' example, every step printed:
H.explain_dcg({"A": 9, "B": 3, "C": 7, "D": 2, "E": 5, "F": 8},
              ["B", "A", "C", "F", "E"])

### 💻 B9. `dcg` and `ndcg` (exercise Problems 5 and 6)
`df_utility` holds only the selected users' rows; `recos` is one list per user, in the same order. DCG is **averaged over users**.

In [ ]:
def dcg(df_utility, recos):
    """Average DCG over the users in df_utility."""
    # YOUR CODE HERE
    raise NotImplementedError


def ndcg(df_utility, recos):
    """DCG divided by the DCG of each user's ideal ordering."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
one = pd.DataFrame([[9, 3, 7, 2, 5, 8]], columns=list("ABCDEF"))
assert_almost_equal(dcg(one, [list("BACFE")]), 514.72, decimal=2)
assert_almost_equal(ndcg(one, [list("BACFE")]), 0.685, decimal=3)
two = pd.DataFrame([[9, 3, 7, 2, 5, 8], [1, 2, 3, 4, nan, 6]],
                   columns=list("ABCDEF"))
assert_almost_equal(dcg(two, [list("BACFE"), list("FEDCB")]),
                    (514.72 + 63 + 0 + 15 / 2 + 7 / math.log2(5)
                     + 3 / math.log2(6)) / 2, decimal=2)
assert_almost_equal(ndcg(one, [list("AFCEB")]), 1.0)
print("✅ dcg / ndcg passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def dcg(df_utility, recos):
    """Average DCG over the users in df_utility."""
    total = 0.0
    for user, rec in zip(df_utility.index, recos):
        rel = df_utility.loc[user, rec].fillna(0).to_numpy(dtype=float)
        ranks = np.arange(1, len(rel) + 1)
        total += np.sum((2 ** rel - 1) / np.log2(ranks + 1))
    return total / len(recos)


def ndcg(df_utility, recos):
    """DCG divided by the DCG of each user's ideal ordering."""
    ideal = []
    for user, rec in zip(df_utility.index, recos):
        best = (df_utility.loc[user].dropna().sort_index()
                .sort_values(ascending=False, kind="mergesort"))
        ideal.append(best.index[:len(rec)].tolist())
    return dcg(df_utility, recos) / dcg(df_utility, ideal)
```

</details>

### 💻 B10. Bonus: RMSE between two matrices
Rating accuracy is graded like regression (MSE, RMSE, MAE, MAPE). Compare only cells present in both.

In [ ]:
def rmse(df_true, df_pred):
    """RMSE over cells that are non-null in both DataFrames."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
t = pd.DataFrame({"a": [4, nan], "b": [2, 5]})
p = pd.DataFrame({"a": [3, 1], "b": [nan, 2]})
assert_almost_equal(rmse(t, p), math.sqrt((1 + 9) / 2))
print("✅ rmse passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def rmse(df_true, df_pred):
    """RMSE over cells that are non-null in both DataFrames."""
    both = df_true.notna() & df_pred.notna()
    err = (df_true - df_pred)[both].stack()
    return float(np.sqrt((err ** 2).mean()))
```

</details>

---
# Part C — Latent Factor Collaborative Filtering

$$\mathbf{M} \approx \mathbf{U}\mathbf{V}^T \qquad \hat{r}_{ij} = \sum_{s=1}^{d} u_{is} v_{js} \qquad J = \frac{1}{2}\sum_{\text{known } (i,j)} \Big(r_{ij} - \sum_{s} u_{is}v_{js}\Big)^2$$

$\mathbf U$ is $n \times d$, $\mathbf V$ is $k \times d$. Storage is $(n + k)d$ numbers instead of $nk$.

In [ ]:
M5 = np.array([[5, 2, 4, 4, 3],
               [3, 1, 2, 4, 1],
               [nan, nan, 3, 1, 4],
               [2, 5, 4, 3, 5],
               [4, 4, 5, 4, nan]])
df5 = pd.DataFrame(M5)
df5

### ✍️ C1. ALS round 1 by hand

Start with $\mathbf U$ = all ones and $d = 2$. Each column's prediction is then $v_{j1} + v_{j2}$. The best single number for a column is **its average known rating**, split evenly between the two factors.

In [ ]:
my = {
    "v_11 (column 1)": None,
    "v_21 (column 2)": None,
    "v_51 (column 5)": None,
    "u_11 after fixing V": None,
}
_c = np.nanmean(M5, axis=0) / 2          # each v_j1 = v_j2
_k = np.isfinite(M5[0])
_s = (M5[0, _k] @ _c[_k]) / (_c[_k] @ _c[_k])   # best u_11 + u_12
check(my, {
    "v_11 (column 1)": _c[0],
    "v_21 (column 2)": _c[1],
    "v_51 (column 5)": _c[4],
    "u_11 after fixing V": _s / 2,
})

In [ ]:
U1, V1 = H.trace_als(M5, 2, rounds=2)   # 🔎 every step printed

### 💻 C2. `als`

In the exercise this was given, but the exam could ask you to write it. Start both matrices at all ones; alternate: fix $\mathbf U$, solve each row of $\mathbf V$ by least squares on that item's known ratings; fix $\mathbf V$, solve each row of $\mathbf U$. Stop when RMSE on known ratings < `tol`.

In [ ]:
def als(M, d, tol):
    """Alternating least squares; returns (U, V) once RMSE < tol."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
U_a, V_a = als(M5, 2, 10)             # huge tol -> stops after round 1
assert_array_almost_equal(V_a[:, 0], [1.75, 1.5, 1.8, 1.6, 1.625])
assert_array_almost_equal(
    U_a[:, 0], [1.09884117, 0.66802999, 0.79970381, 1.13156101,
                1.27784027])
U_b, V_b = als(M5, 2, 1.0)
assert np.sqrt(np.nanmean((M5 - U_b @ V_b.T) ** 2)) < 1.0
print("✅ als passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def als(M, d, tol):
    """Alternating least squares; returns (U, V) once RMSE < tol."""
    M = np.asarray(M, dtype=float)
    U = np.ones((M.shape[0], d))
    V = np.ones((M.shape[1], d))
    while True:
        for j in range(V.shape[0]):
            has = np.isfinite(M[:, j])
            V[j, :] = lstsq(U[has, :], M[has, j])[0]
        for i in range(U.shape[0]):
            has = np.isfinite(M[i, :])
            U[i, :] = lstsq(V[has, :], M[i, has])[0]
        err = np.sqrt(np.nanmean((M - U @ V.T) ** 2))
        if err < tol:
            return U, V
```

</details>

### 💻 C3. `recommend_als` (exercise Problem 1)

⚠️ `user` is a **position** here: use `f_user[user]` and `df_utility.iloc[user]`, not `.loc`. Predicted row = `f_user[user] @ f_item.T`. Keep unrated items, sort by score (high → low) then item id.

In [ ]:
def recommend_als(user, df_utility, f_user, f_item, L):
    """Top-L unrated items for the user at position `user`."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert_equal(recommend_als(2, df5, U_a, V_a, 2), [0, 1])
assert_equal(recommend_als(4, df5, U_a, V_a, 5), [4])
tie = pd.DataFrame([[nan, nan, nan]])
assert_equal(recommend_als(0, tie, np.array([[1.0]]),
                           np.array([[2.0], [1.0], [2.0]]), 3), [0, 2, 1])
print("✅ recommend_als passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def recommend_als(user, df_utility, f_user, f_item, L):
    """Top-L unrated items for the user at position `user`."""
    preds = pd.Series(f_user[user] @ f_item.T, index=df_utility.columns)
    preds = preds[df_utility.iloc[user].isna().to_numpy()]
    ranked = (preds.sort_index()
              .sort_values(ascending=False, kind="mergesort"))
    return ranked.index[:L].tolist()
```

</details>

### ✍️ C4. Coordinate descent by hand

Freeze **everything except one number**, find its best value exactly, move on. Start: $\mathbf U = \mathbf V$ = all ones ($d = 2$), so every prediction is 2.

- Let $u_{11} = x$. Row 1 becomes $x + 1$ against 5, 2, 4, 4, 3. Minimize $(4-x)^2+(1-x)^2+(3-x)^2+(3-x)^2+(2-x)^2$.
- Then with $u_{11} = 2.6$, let $v_{11} = y$. Column 1 (known: 5, 3, 2, 4) gives $(4-2.6y)^2+(2-y)^2+(1-y)^2+(3-y)^2$.

General update (Problem 2 derives the $v$ version the same way):

$$u_{ij} = \frac{\sum_{s} v_{sj}\big(r_{is} - \sum_{t \ne j} u_{it}v_{st}\big)}{\sum_{s} v_{sj}^2} \qquad v_{ij} = \frac{\sum_{s} u_{sj}\big(r_{si} - \sum_{t \ne j} u_{st}v_{it}\big)}{\sum_{s} u_{sj}^2}$$

In [ ]:
my = {"x = u_11": None, "y = v_11": None}
_U, _V = np.ones((5, 2)), np.ones((5, 2))
_U[0, 0] = _x = H.best_u(M5, _U, _V, 0, 0)
check(my, {"x = u_11": _x, "y = v_11": H.best_v(M5, _U, _V, 0, 0)})

In [ ]:
# 🔎 The exercise's update ORDER: row 1 of U (left to right), row 2 ...
# then the rows of V. (The slides' y step jumps ahead to v_11.)
H.trace_cd(M5, 2, steps=4);

### 💻 C5. Single-element updates (the core of `cd`)

In [ ]:
def update_u(M, U, V, i, j):
    """Best value of U[i, j] with everything else held fixed."""
    # YOUR CODE HERE
    raise NotImplementedError


def update_v(M, U, V, i, j):
    """Best value of V[i, j] with everything else held fixed."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
U0, V0 = np.ones((5, 2)), np.ones((5, 2))
x = update_u(M5, U0, V0, 0, 0)
assert_almost_equal(x, 2.6)
U0[0, 0] = x
assert_almost_equal(update_v(M5, U0, V0, 0, 0), 1.6803, decimal=4)
assert_almost_equal(update_u(M5, U0, V0, 0, 1), 1.0)
print("✅ update_u / update_v passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def update_u(M, U, V, i, j):
    """Best value of U[i, j] with everything else held fixed."""
    has = np.isfinite(M[i, :])                    # items user i rated
    rest = U[i] @ V[has].T - U[i, j] * V[has, j]  # other factors' part
    return (V[has, j] @ (M[i, has] - rest)) / (V[has, j] @ V[has, j])


def update_v(M, U, V, i, j):
    """Best value of V[i, j] with everything else held fixed."""
    has = np.isfinite(M[:, i])                    # users who rated i
    rest = U[has] @ V[i] - U[has, j] * V[i, j]
    return (U[has, j] @ (M[has, i] - rest)) / (U[has, j] @ U[has, j])
```

</details>

### 💻 C6. `cd` (exercise Problem 3)

All ones to start; sweep rows of `f_user` (left to right), then rows of `f_item`. Stop when the **fractional SSE improvement** $(\text{SSE}_{prev} - \text{SSE}_{now}) / \text{SSE}_{prev}$ drops below `tol`.

In [ ]:
def cd(df_utility, k, tol):
    """Coordinate descent; returns (f_user, f_item)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
f_user, f_item = cd(df5, 2, 0.01)
assert_equal(f_user.shape, (5, 2))
assert_equal(f_item.shape, (5, 2))
assert_array_almost_equal(f_user[0], [2.2704, 1.6349], decimal=3)
assert_array_almost_equal(f_item[0], [0.7482, 1.7839], decimal=3)
print("✅ cd passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def cd(df_utility, k, tol):
    """Coordinate descent; returns (f_user, f_item)."""
    M = np.asarray(df_utility, dtype=float)
    U = np.ones((M.shape[0], k))
    V = np.ones((M.shape[1], k))
    sse = lambda: np.nansum((M - U @ V.T) ** 2)
    prev = sse()
    while True:
        for i in range(U.shape[0]):
            for j in range(k):
                U[i, j] = update_u(M, U, V, i, j)
        for i in range(V.shape[0]):
            for j in range(k):
                V[i, j] = update_v(M, U, V, i, j)
        cur = sse()
        if (prev - cur) / prev < tol:
            return U, V
        prev = cur
```

</details>

### 💻 C7. `recommend_cd` (exercise Problem 4)

In [ ]:
def recommend_cd(user, df_utility, f_user, f_item, N):
    """Top-N unrated items from the coordinate-descent factors."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert_equal(recommend_cd(2, df5, f_user, f_item, 2), [1, 0])
print("✅ recommend_cd passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def recommend_cd(user, df_utility, f_user, f_item, N):
    """Top-N unrated items from the coordinate-descent factors."""
    return recommend_als(user, df_utility, f_user, f_item, N)
```

</details>

### 💻 C8. Bonus: batch gradient descent

$$e_{ij} = r_{ij} - \sum_s u_{is}v_{js} \qquad \mathbf U \leftarrow \mathbf U + \alpha\mathbf{E}\mathbf V \qquad \mathbf V \leftarrow \mathbf V + \alpha\mathbf{E}^T\mathbf U$$

$\mathbf E$ holds errors on **known** cells and 0 elsewhere. Compute both new matrices from the **old** ones, then swap them in together.

In [ ]:
def gd(M, d, alpha, n_iter, seed=0):
    """Batch gradient descent from a random start; returns (U, V)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
U_g, V_g = gd(M5, 2, 0.01, 2000)
fit = np.sqrt(np.nanmean((M5 - U_g @ V_g.T) ** 2))
assert fit < 0.5, fit
print(f"✅ gd passed (RMSE on known ratings = {fit:.3f})")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def gd(M, d, alpha, n_iter, seed=0):
    """Batch gradient descent from a random start; returns (U, V)."""
    M = np.asarray(M, dtype=float)
    rng = np.random.default_rng(seed)
    U = rng.random((M.shape[0], d))
    V = rng.random((M.shape[1], d))
    known = np.isfinite(M)
    Mz = np.where(known, M, 0.0)
    for _ in range(n_iter):
        E = (Mz - U @ V.T) * known
        U, V = U + alpha * E @ V, V + alpha * E.T @ U
    return U, V
```

</details>

### 💻 C9. Surprise prep: wide → long table

Surprise needs one row per rating: `(user, item, rating)`, blanks dropped. Then:
```python
from surprise import Reader, Dataset, KNNWithMeans
knn = KNNWithMeans(k=5, sim_options={"name": "pearson", "user_based": True})
data = Dataset.load_from_df(to_long(df), Reader(rating_scale=(-10, 10)))
knn.fit(data.build_full_trainset())
knn.predict(0, 1)                                    # user 0, item 1
knn.test(knn.trainset.build_anti_testset())          # all UNRATED pairs
```
Set `"user_based": False` for item-based.

In [ ]:
def to_long(df_utility):
    """Wide utility matrix -> DataFrame[userID, itemID, rating]."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
long = to_long(df_utility)
assert_equal(list(long.columns), ["userID", "itemID", "rating"])
assert_equal(len(long), 11)
assert ("C", "SW2", 5.0) in set(map(tuple, long.to_numpy()))
print("✅ to_long passed")

<details><summary>🔑 <b>Show solution</b></summary>

```python
def to_long(df_utility):
    """Wide utility matrix -> DataFrame[userID, itemID, rating]."""
    return (df_utility.rename_axis("userID").reset_index()
            .melt("userID", var_name="itemID", value_name="rating")
            .dropna().reset_index(drop=True))
```

</details>

In [ ]:
try:
    from surprise import Reader, Dataset, KNNWithMeans
    data = Dataset.load_from_df(to_long(df_six),
                                Reader(rating_scale=(1, 7)))
    knn = KNNWithMeans(k=3, sim_options={"name": "pearson",
                                         "user_based": True},
                       verbose=False)
    knn.fit(data.build_full_trainset())
    print("Surprise guess for B, c:", knn.predict("B", "c").est)
except ImportError:
    print("Surprise not installed here: pip install scikit-surprise")

---
# Part D — Original exercise tests on Jester (course JupyterHub only)

Uses your functions from above. If the file isn't there, it just says so.

In [ ]:
from pathlib import Path

JESTER = Path("/mnt/data/public/jester/dataset1/jester-data-2.xls")
if not JESTER.exists():
    print("Jester file not found: run this on the course JupyterHub.")
else:
    df_jester = pd.read_excel(JESTER, header=None, nrows=100).iloc[:, 1:]
    df_jester.replace(99, np.nan, inplace=True)

    uc = user_complete(df_jester, 5)
    assert_array_almost_equal(uc.iloc[0, :5].tolist(),
                              [-3.446504, 8.11, -1.135967, -2.830721,
                               -2.28])
    assert_array_almost_equal(uc.iloc[13, :5].tolist(),
                              [np.nan, 5.8720833333333315, np.nan,
                               np.nan, -0.49])
    assert_array_equal(recommend(5, df_jester, uc, 10),
                       [77, 45, 80, 11, 98, 52, 25, 85, 10, 39])
    ic = item_complete(df_jester, 5)
    assert_array_almost_equal(ic.iloc[0, :5].tolist(),
                              [7.09, 8.11, -7.28, np.nan, -2.28])
    recos5 = [8, 61, 17, 42, 49, 27, 35, 7, 48, 66]
    recos19 = [7, 70, 32, 5, 18, 26, 53, 14, 97, 17]
    assert_almost_equal(dcg(df_jester.loc[[5]], [recos5]),
                        23.023067518292404)
    assert_almost_equal(ndcg(df_jester.loc[[5, 19]], [recos5, recos19]),
                        0.10609066993151361)
    fu, fi_ = als(df_jester, 50, 2)
    assert_equal(recommend_als(5, df_jester, fu, fi_, 10),
                 [51, 55, 1, 70, 30, 63, 9, 52, 43, 59])
    fu, fi_ = cd(df_jester, 5, 0.01)
    assert_array_almost_equal(fu[0, :], [-5.54176877, 1.11810181,
                                         0.50455207, 2.50592182,
                                         0.80730106])
    assert_array_equal(recommend_cd(0, df_jester, fu, fi_, 10),
                       [83, 73, 72, 80, 85, 81, 100, 98, 90, 51])
    print("✅ all original exercise tests passed")

---
# Part E — Exam-day cheat sheet

**Sorting and ties**
```python
# FIM output: support desc, size desc, alphabetical
sorted(d.items(), key=lambda kv: (-kv[1], -len(kv[0]), kv[0]))
# pandas: score desc, ties by id asc (stable sort after sort_index)
s.sort_index().sort_values(ascending=False, kind="mergesort")
# DataFrame version
df.sort_values(["score", "item"], ascending=[False, True])
```

**Sets and combinations**
```python
set(t).issuperset(combo)          # does receipt t contain combo?
itertools.combinations(items, k)  # all k-sized groups, in sorted order
set(a) < set(b)                   # proper subset (for maximal itemsets)
```

**Ratings tables**
```python
df.mean(axis=1)                   # row means, NaN skipped automatically
df.sub(df.mean(axis=1), axis=0)   # mean-center each ROW
a.notna() & b.notna()             # items both rated
df.columns[df.loc[u].isna()]      # u's unrated items
np.isfinite(M[:, j])              # numpy mask of known ratings
np.nanmean((M - U @ V.T) ** 2)    # MSE on known cells only
```

**Traps that cost points**
- `.loc` (label) vs `.iloc` (position). `recommend` uses `.loc[user]`; `recommend_als` uses `f_user[user]` and `.iloc[user]`.
- Jester columns are labeled 1–100 after `.iloc[:, 1:]`; rows are 0–99.
- Exercise user-based CF: top k first, **then** keep those who rated the item. Weight sum 0 → NaN.
- Item-based similarity still centers by the **user** mean.
- DCG gain is $2^r - 1$; the discount starts at $\log_2(1+1) = 1$.
- pyFIM `supp`: positive = percent, negative = count.
- Lines ≤ 79 characters.